In [0]:
from pyspark.sql import SparkSession

# Récupère la session Spark active
spark = SparkSession.builder.getOrCreate()

# Récupère dbutils de façon compatible (Databricks SDK / PySpark)
try:
    from databricks.sdk.runtime import dbutils
except ImportError:
    try:
        from pyspark.dbutils import DBUtils
        dbutils = DBUtils(spark)
    except ImportError:
        pass  # En environnement Databricks natif, dbutils est déjà injecté

# ============================================================
# CONFIGURATION
# ============================================================

# Nom du catalog UC existant (à vérifier avec SHOW CATALOGS)
CATALOG = "main"

SCHEMAS = ["bronze", "silver", "gold"]
VOLUME_SUFFIX = "_volume"

# ============================================================
# FONCTIONS UTILITAIRES
# ============================================================


def log(msg):
    print(f"[SETUP] {msg}")


def catalog_exists(name):
    catalogs = [row.catalog for row in spark.sql("SHOW CATALOGS").collect()]
    return name in catalogs


def create_schema(catalog, schema):
    log(f"Création du schema UC : {catalog}.{schema}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")


def create_volume(catalog, schema, volume_name):
    log(f"Création du volume : {catalog}.{schema}.{volume_name}")
    spark.sql(f"USE CATALOG {catalog}")
    spark.sql(f"USE SCHEMA {schema}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {volume_name}")


def create_directories(base_path):
    log(f"Création des dossiers techniques dans : {base_path}")
    dbutils.fs.mkdirs(f"{base_path}/landing_zone")
    dbutils.fs.mkdirs(f"{base_path}/kaggle")
    dbutils.fs.mkdirs(f"{base_path}/_checkpoints")

# ============================================================
# EXECUTION
# ============================================================


log("Vérification du catalog UC...")

if not catalog_exists(CATALOG):
    raise Exception(f"Catalog UC '{CATALOG}' introuvable. Vérifie avec SHOW CATALOGS.")

log(f"Catalog UC détecté : {CATALOG}")

for schema in SCHEMAS:
    volume_name = schema + VOLUME_SUFFIX

    # 1. Création du schema UC
    create_schema(CATALOG, schema)

    # 2. Création du volume UC
    create_volume(CATALOG, schema, volume_name)

    # 3. Construction du chemin du volume
    base_path = f"/Volumes/{CATALOG}/{schema}/{volume_name}"

    # 4. Création des dossiers techniques
    create_directories(base_path)

    log(f"✓ {schema} prêt → {base_path}")

log("✓ Initialisation complète du Lakehouse bronze/silver/gold.")

#################################################
# Pour la partie ML
#################################################

# Création du Schéma et du Volume pour stocker les modèles ML de prédiction
create_schema(CATALOG, "ml")
create_volume(CATALOG, "ml", "models_volume")
dbutils.fs.mkdirs(f"/Volumes/{CATALOG}/ml/models_volume/tmp")

log("✓ Création du dossier pour le ML")